In [ ]:
#Imports data

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, 
                              accuracy_score, f1_score)
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

All imports successful!


In [7]:
#Load Data
import sys
import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.append('..')

# Load the processed features (from Dinuka's preprocessing)
df_features = pd.read_csv('../data/final_processed_dataset.csv')

# Load the original raw data to get the target label
df_raw = pd.read_csv('../data/pokemon_complete_2025.csv')

# Check what columns are in the raw file that could be the target
print("Raw CSV columns:")
print(df_raw.columns.tolist())

Raw CSV columns:
['pokedex_id', 'name', 'genus', 'generation', 'type_1', 'type_2', 'num_types', 'hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed', 'base_stat_total', 'height_m', 'weight_kg', 'base_experience', 'ability_1', 'ability_2', 'hidden_ability', 'color', 'shape', 'habitat', 'growth_rate', 'egg_groups', 'is_legendary', 'is_mythical', 'is_baby', 'capture_rate', 'base_happiness', 'hatch_counter', 'gender_rate', 'description', 'sprite_url', 'is_dual_type', 'bmi', 'attack_defense_ratio', 'physical_total', 'special_total', 'offensive_total', 'defensive_total', 'gender_distribution', 'stat_tier']


In [8]:
import pandas as pd

df_raw = pd.read_csv('../data/pokemon_complete_2025.csv')
print("Shape:", df_raw.shape)
print("\nAll columns:")
for col in df_raw.columns.tolist():
    print(col)

Shape: (1025, 43)

All columns:
pokedex_id
name
genus
generation
type_1
type_2
num_types
hp
attack
defense
sp_attack
sp_defense
speed
base_stat_total
height_m
weight_kg
base_experience
ability_1
ability_2
hidden_ability
color
shape
habitat
growth_rate
egg_groups
is_legendary
is_mythical
is_baby
capture_rate
base_happiness
hatch_counter
gender_rate
description
sprite_url
is_dual_type
bmi
attack_defense_ratio
physical_total
special_total
offensive_total
defensive_total
gender_distribution
stat_tier


In [10]:
print("stat_tier unique values:")
print(df_raw['stat_tier'].value_counts())

stat_tier unique values:
stat_tier
Average (400-499)          329
Strong (500-599)           250
Below Average (300-399)    240
Weak (<300)                146
Legendary/Pseudo (600+)     60
Name: count, dtype: int64


In [11]:
print(df_raw.columns.tolist())

['pokedex_id', 'name', 'genus', 'generation', 'type_1', 'type_2', 'num_types', 'hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed', 'base_stat_total', 'height_m', 'weight_kg', 'base_experience', 'ability_1', 'ability_2', 'hidden_ability', 'color', 'shape', 'habitat', 'growth_rate', 'egg_groups', 'is_legendary', 'is_mythical', 'is_baby', 'capture_rate', 'base_happiness', 'hatch_counter', 'gender_rate', 'description', 'sprite_url', 'is_dual_type', 'bmi', 'attack_defense_ratio', 'physical_total', 'special_total', 'offensive_total', 'defensive_total', 'gender_distribution', 'stat_tier']


In [12]:
# Open and read models.py to understand what target was used
with open('../src/models.py', 'r') as f:
    print(f.read())

from __future__ import annotations

from pathlib import Path
from typing import Dict

import joblib
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def train_random_forest(
    X_train,
    y_train,
    *,
    n_estimators: int = 300,
    random_state: int = 42,
) -> RandomForestRegressor:
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        random_state=random_state,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model


def evaluate_regressor(model, X_test, y_test) -> Dict[str, float]:
    preds = model.predict(X_test)
    rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
    mae = float(mean_absolute_error(y_test, preds))
    r2 = float(r2_score(y_test, preds))
    return {"rmse": rmse, "mae": mae, "r2": r2}


def save_model(model, model_path: str | Path) -> None:
    path = Path(model_path)
    path.parent.mkdir(parents=True, exi

In [13]:
import joblib

lr_model = joblib.load('../models/balance_risk_lr_pipeline.joblib')
print(type(lr_model))
print(lr_model)

# If it's a pipeline, check steps
if hasattr(lr_model, 'steps'):
    print("\nPipeline steps:")
    for step in lr_model.steps:
        print(step)

# Check what classes it predicts
if hasattr(lr_model, 'classes_'):
    print("\nClasses:", lr_model.classes_)

<class 'sklearn.pipeline.Pipeline'>
Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(C=10, class_weight='balanced',
                                    max_iter=1000, random_state=42))])

Pipeline steps:
('scaler', StandardScaler())
('classifier', LogisticRegression(C=10, class_weight='balanced', max_iter=1000,
                   random_state=42))

Classes: [0 1]


In [14]:
# Check the binary columns to find what maps to 0/1 balance risk
print("is_legendary:", df_raw['is_legendary'].value_counts())
print("\nis_mythical:", df_raw['is_mythical'].value_counts())

# Try creating the likely target: uber = legendary OR mythical
df_raw['is_uber'] = ((df_raw['is_legendary'] == True) | (df_raw['is_mythical'] == True)).astype(int)
print("\nis_uber (created):", df_raw['is_uber'].value_counts())

is_legendary: is_legendary
False    954
True      71
Name: count, dtype: int64

is_mythical: is_mythical
False    1002
True       23
Name: count, dtype: int64

is_uber (created): is_uber
0    931
1     94
Name: count, dtype: int64


In [15]:
import joblib
from sklearn.preprocessing import StandardScaler

lr_model = joblib.load('../models/balance_risk_lr_pipeline.joblib')

# Use only numeric columns as features (matching what the model was trained on)
feature_cols = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed',
                'base_stat_total', 'height_m', 'weight_kg', 'base_experience',
                'capture_rate', 'base_happiness', 'hatch_counter', 'gender_rate',
                'bmi', 'attack_defense_ratio', 'physical_total', 'special_total',
                'offensive_total', 'defensive_total']

X = df_raw[feature_cols].fillna(0)
y = df_raw['is_uber']

# Quick test prediction
sample_preds = lr_model.predict(X[:10])
print("Sample predictions:", sample_preds)
print("Actual labels:     ", y[:10].values)

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- base_stat_total
- offensive_total
Feature names seen at fit time, yet now missing:
- num_types
- type_bug
- type_dark
- type_dragon
- type_electric
- ...


In [17]:
#Corrected version of data loading cell
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load processed features (what the model was trained on)
df_features = pd.read_csv('../data/final_processed_dataset.csv')

# Load raw data to create the target label
df_raw = pd.read_csv('../data/pokemon_complete_2025.csv')

# Create binary target: 1 = Uber/Broken (legendary or mythical), 0 = Balanced
df_raw['is_uber'] = ((df_raw['is_legendary'] == True) | (df_raw['is_mythical'] == True)).astype(int)
y = df_raw['is_uber']

# Features = processed dataset BUT remove columns the model hasn't seen
# Based on the error: model expects num_types + type_* columns, NOT base_stat_total/offensive_total
X = df_features.drop(columns=['pokedex_id'], errors='ignore')

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X columns:", X.columns.tolist())
print("\nTarget distribution:")
print(y.value_counts())

# 80/20 split (matching Sumudu's training split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

X shape: (1025, 39)
y shape: (1025,)
X columns: ['num_types', 'hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed', 'base_stat_total', 'height_m', 'weight_kg', 'base_experience', 'capture_rate', 'base_happiness', 'hatch_counter', 'gender_rate', 'bmi', 'attack_defense_ratio', 'physical_total', 'special_total', 'offensive_total', 'defensive_total', 'type_bug', 'type_dark', 'type_dragon', 'type_electric', 'type_fairy', 'type_fighting', 'type_fire', 'type_flying', 'type_ghost', 'type_grass', 'type_ground', 'type_ice', 'type_normal', 'type_poison', 'type_psychic', 'type_rock', 'type_steel', 'type_water']

Target distribution:
is_uber
0    931
1     94
Name: count, dtype: int64

Train: 820 samples
Test:  205 samples


In [18]:
import joblib

lr_model = joblib.load('../models/balance_risk_lr_pipeline.joblib')

# Quick prediction test
sample_preds = lr_model.predict(X_test[:10])
print("Sample predictions:", sample_preds)
print("Actual labels:     ", y_test[:10].values)

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- base_stat_total
- offensive_total


In [19]:
# Get the exact feature names the model was trained on
print("Features model expects:")
print(lr_model[0].feature_names_in_.tolist())
print("\nCount:", len(lr_model[0].feature_names_in_))

Features model expects:
['num_types', 'hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed', 'height_m', 'weight_kg', 'base_experience', 'capture_rate', 'base_happiness', 'hatch_counter', 'gender_rate', 'bmi', 'attack_defense_ratio', 'physical_total', 'special_total', 'defensive_total', 'type_bug', 'type_dark', 'type_dragon', 'type_electric', 'type_fairy', 'type_fighting', 'type_fire', 'type_flying', 'type_ghost', 'type_grass', 'type_ground', 'type_ice', 'type_normal', 'type_poison', 'type_psychic', 'type_rock', 'type_steel', 'type_water']

Count: 37


In [20]:
# Use only the exact columns the model was trained on
expected_features = lr_model[0].feature_names_in_.tolist()

X = df_features[expected_features]

# Redo the split with correct X
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Test prediction again
sample_preds = lr_model.predict(X_test[:10])
print("Sample predictions:", sample_preds)
print("Actual labels:     ", y_test[:10].values)

Sample predictions: [0 0 0 0 0 0 0 0 0 0]
Actual labels:      [0 0 0 0 0 0 0 0 0 0]
